[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI/blob/main/notebooks/04_rag_pipeline.ipynb)

**Where to run this notebook**

- **Locally (VS Code + Jupyter)**: just open the notebook and pick the `dsai-m3` kernel. Your `.env` file (see `setup.md`) applies the macOS OpenMP fix automatically.
- **Colab (recommended if you don't have a GPU)**: click the badge above, then **Runtime → Change runtime type → T4 GPU**, then run the setup cell below. It clones the repo, installs missing deps, and `cd`s into the right working directory.


In [25]:
# === Colab-compat setup (no-op when running locally) ===
# This whole `if IN_COLAB:` block does NOTHING on your laptop — it only fires
# on Google Colab, where it clones the course repo, installs the missing
# libraries, and moves into the right folder so the data/ paths resolve.
import os, sys

IN_COLAB = "google.colab" in sys.modules  # True only inside a Colab runtime

if IN_COLAB:
    REPO_URL = "https://github.com/su-ntu-ctp/6m-data-3.10-Transformers-Attention-GenAI.git"
    REPO_DIR = "/content/6m-data-3.10-Transformers-Attention-GenAI"
    LESSON_DIR = "notebooks"

    if not os.path.exists(REPO_DIR):
        print(f"Cloning repo into {REPO_DIR} ...")
        os.system(f"git clone -q {REPO_URL} {REPO_DIR}")

    os.chdir(f"{REPO_DIR}/{LESSON_DIR}")
    print(f"Working directory: {os.getcwd()}")

    # Colab has torch + torchvision pre-installed. Install the rest.
    os.system("pip install -q sentence-transformers transformers")
    print("Colab setup done.")

# Threading caps — picked up by .env locally, set explicitly here as a fallback.
# Harmless if already set. (Loop form prevents Jupyter from auto-displaying the return value.)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")   # silence a harmless warning
# (Old OMP/MKL one-thread caps removed — they slowed CPU-only machines.)

'false'

# L10 · NB 04 — RAG: Retrieval-Augmented Generation

> ⏱️ **~35 min** core · +15 min for the 🟡 Extension · slow on CPU? use the Colab T4 badge above

> *We have embeddings (L09). We have an LLM (NB 03). Let's stitch them together into the architecture behind every modern "chat with your data" product — including Marcus's shopping assistant.*

In this notebook we will:

1. Show why an LLM **alone** can't answer questions about NorthStar's products (it doesn't know them — it hallucinates)
2. Build a retrieval step using `all-MiniLM-L6-v2` (from L09)
3. Stuff retrieved products into a prompt and send it to SmolLM2-360M
4. Watch the LLM produce a grounded answer with real product names
5. Build a clean `RAGSystem` class — what you'd actually ship

This is the closing pattern of the entire module.

---

### Where you are in L10

Marcus asked for a shopping assistant. Sarah can't build it in one leap — she works through four notebooks, each answering the next question in the chain:

| Step | Notebook | Sarah's question |
|---|---|---|
| 1 | `01_morning_briefing` | What can pretrained transformers already do, out of the box? |
| 2 | `02_attention_intuition` | *Why* do they work? What is this "attention" everyone talks about? |
| 3 | `03_using_an_llm` | How do I drive a generative LLM myself — tokens, sampling, chat? |
| 4 | `04_rag_pipeline` 👈 **you are here** | How do I ground it in NorthStar's catalogue? → the shopping assistant |

Right now you're on **step 4 of 4**. Each notebook stands on the one before it — run them in order.


## 1 · Setup


> **Why this cell first — and why it looks like a pile of switches.**
> Before Sarah touches a single model she runs this setup cell. It isn't glamorous, but every line earns its place:
>
> - `KMP_DUPLICATE_LIB_OK='TRUE'` — on macOS, PyTorch and the other scientific libraries each ship their own copy of the OpenMP maths runtime. When two copies load at once the kernel **crashes the instant you `import torch`**. This line tells them to coexist instead of fighting. (It's the same fix the repo's `.env` file applies — we repeat it here so the notebook also works in Colab and on a fresh machine.)
> - `HF_HUB_DISABLE_TELEMETRY` / `TRANSFORMERS_VERBOSITY='error'` / `warnings.filterwarnings('ignore')` — silence the download chatter and version warnings so the output you see is the output that *matters*.
>
> You'll see this same opening cell in every model notebook. Run it once at the top — it's the seatbelt before the drive.


In [26]:
# --- Stability + quiet-output environment switches (set BEFORE importing torch) ---
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'  # allow duplicate OpenMP runtime (macOS libomp conflict)
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'  # don't phone home when downloading models
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'  # hide info/warning chatter from transformers

import warnings
warnings.filterwarnings('ignore')  # silence version/deprecation warnings so real output stands out

# --- Core libraries (grouped): numerics, data tables, deep-learning, the two model families ---
import time                                                    # to time how long generation takes
import numpy as np                                             # array math (argsort, indexing)
import pandas as pd                                            # load + slice the catalogue CSV
import torch                                                   # the tensor/deep-learning backend
from sentence_transformers import SentenceTransformer          # the EMBEDDING (retrieval) model
from sklearn.metrics.pairwise import cosine_similarity         # measures how similar two embeddings are
from transformers import AutoTokenizer, AutoModelForCausalLM   # the GENERATIVE LLM + its tokenizer

# (Removed the old one-CPU-thread cap — it made CPU-only machines much slower.)
torch.manual_seed(0)      # fix the random seed so results are reproducible run to run
# Use a GPU if one is available, otherwise fall back to CPU:
#   'cuda' = NVIDIA GPU (e.g. Colab T4), 'mps' = Apple Silicon GPU (M1/M2/M3/M4 Macs)
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'
print(f'Running on: {DEVICE}')

# RAG needs TWO models: one to RETRIEVE (find relevant products), one to GENERATE (write the answer).
print('Loading retrieval model...')
retriever = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)  # turns text into 384-dim vectors (from L09)

print('Loading LLM (SmolLM2-360M-Instruct)...')
LLM_NAME = 'HuggingFaceTB/SmolLM2-360M-Instruct'      # a small instruction-tuned chat model
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)   # converts text <-> token IDs
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME).to(DEVICE)  # the model that predicts the next token
print('Both models ready.')

Running on: mps
Loading retrieval model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading LLM (SmolLM2-360M-Instruct)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Both models ready.


## 2 · Load NorthStar's catalogue

In [27]:
# Load NorthStar's product catalogue from CSV into a pandas DataFrame (a table).
# Each row is one product; columns include name, category, description, price_gbp.
df = pd.read_csv('data/northstar_catalogue.csv')
# .nunique() counts the distinct categories; len(df) is the number of products (rows).
print(f"Catalogue: {len(df)} products across {df['category'].nunique()} categories")
df.head(5)  # peek at the first 5 rows to see the column layout

Catalogue: 76 products across 10 categories


,product_id,name,category,description,price_gbp
0,P0001,Lila Floral Sundress,dress,Lightweight floral frock perfect for warm summ...,49
1,P0002,Marina Wrap Dress,dress,Elegant wrap-style dress in deep navy. Suitabl...,89
2,P0003,Cassia Maxi Gown,dress,Flowing full-length gown with adjustable strap...,65
3,P0004,Holly Knit Jumper Dress,dress,Cosy ribbed knit dress in oatmeal. Perfect win...,72
4,P0005,Ivy Slip Dress,dress,Minimalist satin slip in champagne. Layer over...,55


## 3 · Why an LLM alone fails: the hallucination problem

Let's ask the LLM about NorthStar products WITHOUT giving it the catalogue. It will produce something plausible — but completely invented.

In [28]:
def llm_chat(messages, max_new_tokens=200, do_sample=False, repetition_penalty=1.15):
    """Send a chat-formatted prompt to the LLM.
    repetition_penalty > 1 prevents the model from getting stuck in degenerate
    loops on long contexts — important for a 360M-parameter model handling
    structured catalogue data.
    """
    # Turn the list of role/content messages into the exact text format the model was trained on.
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    # Encode that prompt text into token IDs (a tensor the model can consume).
    input_ids = tokenizer(prompt, return_tensors='pt')['input_ids'].to(DEVICE)
    # Generate new tokens. do_sample=False => greedy/deterministic output.
    out = llm.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=do_sample,
                       pad_token_id=tokenizer.eos_token_id,
                       repetition_penalty=repetition_penalty)
    # Slice off the prompt tokens (input_ids.shape[1]) so we return ONLY the model's new reply.
    return tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()

# Ask the LLM about NorthStar products with no context
question = "I'm looking for a summer dress under £80. What does NorthStar have?"
# Note: there is NO catalogue here — just a system role + the user's question.
naive_messages = [
    {'role': 'system', 'content': 'You are a helpful retail shopping assistant for NorthStar.'},
    {'role': 'user', 'content': question},
]

print(f"Q: {question}\n")
print(f"A (LLM alone, no catalogue):")
t0 = time.time()
# The model has no real product data, so it will INVENT plausible-sounding products (hallucination).
print(llm_chat(naive_messages))
print(f"\n[generated in {time.time()-t0:.1f}s]")

Q: I'm looking for a summer dress under £80. What does NorthStar have?

A (LLM alone, no catalogue):
NorthStar has several stylish and affordable options that fit your criteria:

1. **Luxury Dress**: A high-end, designer dress can cost upwards of £250-$300 per season. However, you might find some more budget-friendly options in the range of £60-$90. 

2. **Midi Dress**: This is a mid-length skirt or blouse with a full hemline. It's often less expensive than a mini dress but still offers a comfortable look. Prices vary from around £40 to £70.

3. **Dress**: If you're not ready for a full length, consider a dress. These usually start at around £30 and go up to £60. They offer versatility as they can be worn over other clothing items like trousers or skirts.

4. **Trendy Dresses**: For a trendier style, try a dress made out of lightweight fabrics such as cotton or

[generated in 20.9s]


**Read that carefully.** The model produced a confident, fluent-sounding response. Almost everything in it is invented — fabricated product names, made-up prices, hallucinated descriptions.

This is the LLM doing exactly what it was trained to do (produce plausible text) with no grounding. It has no idea what NorthStar actually sells.

You CANNOT ship this. A customer who clicks on a recommended product and finds it doesn't exist loses trust forever. We need to fix the hallucination.

## 4 · The fix: retrieve first, then generate

The recipe:
1. Embed the catalogue once (L09)
2. Embed the query
3. Find the top-K most relevant products
4. Include them in the LLM's prompt
5. Ask the LLM to answer ONLY using the retrieved products

In [29]:
# Step 1 + 2: embed the catalogue (once) and the query (per request)
print('Embedding catalogue...')
# Build one descriptive string per product (name + description + price) — richer text embeds better.
docs = (df['name'] + ' — ' + df['description'] + f' (£' + df['price_gbp'].astype(str) + ')').tolist()
# encode() turns each product string into a fixed-length numeric vector capturing its meaning.
catalogue_emb = retriever.encode(docs, show_progress_bar=False)
# Shape is (76, 384): 76 products (rows), each a 384-dimensional embedding (all-MiniLM-L6-v2's size).
print(f"Catalogue embeddings shape: {catalogue_emb.shape}")

Embedding catalogue...
Catalogue embeddings shape: (76, 384)


In [30]:
def retrieve(query, top_k=5, max_price=None):
    """Return the top-K most relevant products as a DataFrame.

    max_price is an optional METADATA FILTER. Embeddings capture meaning, not arithmetic:
    "under £80" in the query text will NOT stop a £95 jacket from scoring well. So when the
    customer gives a budget, we drop over-budget rows explicitly before ranking.
    """
    # Embed the query into the SAME 384-dim space as the catalogue so they're comparable.
    q_emb = retriever.encode([query])
    # cosine_similarity scores the query against every product (0..1); higher = closer in meaning.
    sims = cosine_similarity(q_emb, catalogue_emb)[0]
    out = df.copy()
    out['similarity'] = sims  # attach the score so we can inspect relevance
    if max_price is not None:
        out = out[out['price_gbp'] <= max_price]  # hard budget filter -- applied BEFORE ranking
    # Sort highest similarity first and keep the top_k rows.
    return out.sort_values('similarity', ascending=False).head(top_k).reset_index(drop=True)

# Test the retrieval for our question
question = "I'm looking for a summer dress under £80. What does NorthStar have?"
retrieved = retrieve(question, top_k=5, max_price=80)  # the 5 closest in-budget products
print(f"Top {len(retrieved)} products retrieved:")
print(retrieved[['product_id','name','category','price_gbp','similarity']].to_string(index=False))

Top 5 products retrieved:
product_id                    name category  price_gbp  similarity
     P0007    Sienna Bodycon Dress    dress         45    0.583023
     P0010    Marigold Shift Dress    dress         52    0.561665
     P0004 Holly Knit Jumper Dress    dress         72    0.542383
     P0018    Rose Puff Sleeve Top    shirt         42    0.518634
     P0003        Cassia Maxi Gown    dress         65    0.515624


## 5 · Build the augmented prompt

Format the retrieved products into a system prompt that constrains the LLM's behaviour.

In [31]:
def build_rag_prompt(query, retrieved):
    """Compose a system+user message pair.

    Important design choice: we put the catalogue in the USER turn, not the
    system prompt. A 360M-parameter model can drift on long system prompts
    that contain structured data — it tends to treat them as text to continue
    rather than rules to follow. Putting the catalogue in the user turn,
    framed as "here is what we have, answer my question using only these",
    works much more reliably.
    """
    # Format each retrieved product as one readable bullet line for the prompt.
    product_lines = []
    for _, row in retrieved.iterrows():
        product_lines.append(
            f"- {row['name']} (£{row['price_gbp']}): {row['description']}"
        )
    catalogue_block = '\n'.join(product_lines)  # join all bullets into one block of text

    # The system prompt sets the rules: ONLY the listed products, described ONLY from their descriptions.
    system_prompt = (
        "You are a shopping assistant for NorthStar. Recommend products ONLY from the list "
        "the user gives you. Describe each product using ONLY the facts in its description. "
        "If nothing on the list fits the question, say so."
    )
    # The user turn carries the question PLUS the retrieved catalogue (the "augmented" context).
    user_message = (
        f"Question: {query}\n\n"
        f"Products:\n{catalogue_block}\n\n"
        "Recommend at most two products from the list above, in two sentences."
    )
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': user_message},
    ]

# Show what the prompt looks like
prompt = build_rag_prompt(question, retrieved)  # the grounded prompt we'll feed the LLM
print('System prompt:')
print(prompt[0]['content'])
print('\nUser message:')
print(prompt[1]['content'])

System prompt:
You are a shopping assistant for NorthStar. Recommend products ONLY from the list the user gives you. Describe each product using ONLY the facts in its description. If nothing on the list fits the question, say so.

User message:
Question: I'm looking for a summer dress under £80. What does NorthStar have?

Products:
- Sienna Bodycon Dress (£45): Fitted stretch jersey in midnight black. Goes from desk to drinks effortlessly.
- Marigold Shift Dress (£52): Sleeveless A-line shift in sunshine yellow. Bold colour for the brave.
- Holly Knit Jumper Dress (£72): Cosy ribbed knit dress in oatmeal. Perfect winter layer with thick tights.
- Rose Puff Sleeve Top (£42): Romantic puff-sleeve blouse with smocked waist. Picnic-ready.
- Cassia Maxi Gown (£65): Flowing full-length gown with adjustable straps. Beach holiday essential.

Recommend at most two products from the list above, in two sentences.


## 6 · Send the augmented prompt to the LLM

In [32]:
print(f"Q: {question}\n")
t0 = time.time()
print('A (RAG: retrieve top-5 + ask LLM):\n')
# Same LLM as before, but now its prompt contains the real retrieved products...
answer = llm_chat(prompt, max_new_tokens=120)
print(answer)  # ...so the answer cites genuine NorthStar names/prices instead of inventing them
print(f"\n[generated in {time.time()-t0:.1f}s]")

Q: I'm looking for a summer dress under £80. What does NorthStar have?

A (RAG: retrieve top-5 + ask LLM):

NorthStar has several stylish options that fit your budget of £80 or less. The first is the Sienna Bodycon Dress ($45), which offers a fitted and comfortable look perfect for a summer day. The second is the Marigold Shift Dress ($52) - it's a great choice if you're looking for something more casual but still elegant. Both dresses offer a range of colors suitable for any occasion.

[generated in 12.8s]


**Look at the response.** Every product it names is real, in budget and in the retrieved set — Sienna Bodycon Dress (£45) and Marigold Shift Dress (£52). Compare that with Section 3, where the LLM alone invented a "Midi Dress" line and price ranges NorthStar doesn't sell. Grounding fixed the *what*.

It did not fully fix the *how*. Read closely: the prices come back as `$45` instead of `£45`, and "a range of colors" appears in neither description (Sienna comes in midnight black only). A 360M model still embellishes details, even when the system prompt says *describe each product using ONLY the facts in its description*. That is why the Safety section of Sarah's checklist below insists on validating answers — and why production deployments use a much bigger model (Claude, GPT-4). The *architecture* is the same; the *quality* is what scale buys you.

**Two things to notice in the retrieval table above:**

- **The `max_price=80` filter did real work.** Without it, a £95 Verge Bomber Jacket scores into the top 5. Embeddings capture meaning, not arithmetic — budgets need a metadata filter.
- **The most obvious match is missing.** Lila Floral Sundress (£49, *"perfect for warm summer days"*) describes itself as a *frock*, not a *dress*, and MiniLM ranks it 22nd. If retrieval misses a product, no prompt can recommend it — evaluate retrieval on its own (L09) before tuning the generation step.

This is RAG. The retrieval keeps the model grounded; the LLM does the language understanding and synthesis.

## 7 · Wrap it up: a clean RAGSystem class

In [33]:
class RAGSystem:
    """Minimum-viable RAG shopping assistant."""

    def __init__(self, df, retriever, llm, tokenizer):
        # Store the catalogue and both models on the instance so methods can reuse them.
        self.df = df.reset_index(drop=True).copy()
        self.retriever = retriever
        self.llm = llm
        self.tok = tokenizer
        # Pre-compute catalogue embeddings once
        # (done in __init__ so we embed the catalogue a single time, not on every question).
        docs = (self.df['name'] + ' — ' + self.df['description']
                + ' (£' + self.df['price_gbp'].astype(str) + ')').tolist()
        self.embeddings = self.retriever.encode(docs, show_progress_bar=False)

    def retrieve(self, query, top_k=5, max_price=None):
        # Same retrieval logic as the standalone function: embed query, score, filter, take top_k.
        q_emb = self.retriever.encode([query])
        sims = cosine_similarity(q_emb, self.embeddings)[0]   # similarity to every product
        out = self.df.assign(similarity=sims)
        if max_price is not None:
            out = out[out['price_gbp'] <= max_price]          # metadata filter: drop over-budget rows
        return out.sort_values('similarity', ascending=False).head(top_k).reset_index(drop=True)

    def _build_prompt(self, query, retrieved):
        # Catalogue goes in the USER turn (see build_rag_prompt above for why).
        lines = [f"- {r['name']} (£{r['price_gbp']}): {r['description']}"
                 for _, r in retrieved.iterrows()]
        block = '\n'.join(lines)
        # Same rules as build_rag_prompt: listed products only, facts from the descriptions only.
        sys_msg = (
            "You are a shopping assistant for NorthStar. Recommend products ONLY from the list "
            "the user gives you. Describe each product using ONLY the facts in its description. "
            "If nothing on the list fits the question, say so."
        )
        user_msg = (
            f"Question: {query}\n\n"
            f"Products:\n{block}\n\n"
            "Recommend at most two products from the list above, in two sentences."
        )
        return [
            {'role': 'system', 'content': sys_msg},
            {'role': 'user',   'content': user_msg},
        ]

    def ask(self, query, top_k=5, max_price=None, max_new_tokens=120):
        # The full RAG flow in one call: RETRIEVE -> build prompt -> GENERATE.
        retrieved = self.retrieve(query, top_k=top_k, max_price=max_price)
        messages = self._build_prompt(query, retrieved)
        prompt = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        input_ids = self.tok(prompt, return_tensors='pt')['input_ids'].to(DEVICE)
        out = self.llm.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=self.tok.eos_token_id,
                                repetition_penalty=1.15)
        answer = self.tok.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True).strip()
        # Return BOTH the answer and the retrieved products (the audit trail / grounding).
        return {'answer': answer, 'retrieved': retrieved}

# Instantiate and try a few questions
rag = RAGSystem(df, retriever, llm, tokenizer)  # __init__ embeds the catalogue right here
print('RAGSystem ready.')

RAGSystem ready.


### Try several customer questions

In [34]:
# A few different shopping intents to show retrieval adapts to each query.
# Each is (question, budget) -- the budget feeds the max_price metadata filter (None = no limit).
questions = [
    ("I need something cosy to wear hiking on a cold day", None),
    ("What's a smart shirt I can wear to the office?", None),
    ("Recommend a beach holiday outfit for under £150", 150),
]

for q, budget in questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    t0 = time.time()
    result = rag.ask(q, max_price=budget)  # runs the full retrieve-then-generate pipeline for this question
    print(f"\nRetrieved products (everything the LLM was shown):")
    # Show ALL retrieved items -- this is the audit trail. Hiding some would make a correctly
    # grounded answer look like a hallucination.
    for _, row in result['retrieved'].iterrows():
        print(f"  - {row['name']:<35s} ({row['category']:<10s} £{row['price_gbp']:>3d})")
    print(f"\nA: {result['answer']}")
    print(f"\n[total: {time.time()-t0:.1f}s]")


Q: I need something cosy to wear hiking on a cold day

Retrieved products (everything the LLM was shown):
  - Frost Linen Shirt                   (shirt      £ 55)
  - Aspen Snow Boots                    (footwear   £155)
  - Boulder Hiking Boots                (footwear   £175)
  - Pine Flannel Shirt                  (shirt      £ 48)
  - Tarn Waxed Jacket                   (coat       £145)

A: I would recommend the Frost Linen Shirt as your first choice because it's breathable, lightweight, and perfect for chilly days like this one. The Aspen Snow Boots offer excellent insulation against the elements while being comfortable enough for casual hikes. The Boulder Hiking Boots provide exceptional durability and weather protection, making them ideal for tackling more rugged trails.

[total: 8.4s]

Q: What's a smart shirt I can wear to the office?

Retrieved products (everything the LLM was shown):
  - Indigo Polo Shirt                   (shirt      £ 50)
  - Atlas Oxford Shirt          

**This is RAG in production form.** ~50 lines of Python wrapping two open-source models. It can answer arbitrary natural-language shopping questions, grounded in actual product data, with audit trails (you can see which products were retrieved for any given answer).

This is the architecture behind:
- ChatGPT's "browsing" / "memory"
- Perplexity AI
- Most enterprise "chat with your docs" products
- Customer-support copilots
- Code-completion tools (the docs are the codebase)

Same pattern, different data.

## 8 · Sarah's production checklist

For a production RAG system at NorthStar, Sarah's notes:

**Sarah's production checklist** — what it takes to run this RAG system for real at NorthStar:

**▸ Retrieval**
- `top_k = 5 to 10` — too few misses good products, too many dilutes the prompt
- Hybrid (TF-IDF + semantic) — same as L09
- Apply category/price filters BEFORE retrieval where the user is specific

**▸ Prompt design**
- System prompt MUST say *"only recommend from the catalogue below"*
- System prompt MUST say *"if nothing matches, say so"* — prevents over-eagerness
- Show product IDs so users can click through

**▸ Safety**
- Post-process: extract product names from the answer, validate against the catalogue
- If the answer mentions a product NOT in the retrieved set: flag for review
- Always show the source products alongside the answer (audit trail)

**▸ Quality**
- Log `(query → retrieved → answer)` tuples for offline evaluation
- Hand-label 100 queries as good/bad, track that metric weekly
- A/B test against the L09 keyword+semantic baseline before launch

**▸ Cost / latency**
- Self-hosted SmolLM2: free, but each answer takes several seconds on a laptop (see the timings printed above)
- Hosted API (Claude Haiku, GPT-4 mini): ~$0.001 per query, 500ms latency, much better quality
- Most production setups use a hosted API for quality; self-host as fallback

## 9 · The Module 3 closing recap

What you've built across ten lessons:

| Lesson | Foundational skill |
|--------|--------------------|
| L01-L03 | Classical ML modelling, evaluation, making it work |
| L04 | Trees, ensembles, hyperparameter tuning |
| L05 | Clustering, dimensionality reduction, anomaly detection |
| L06 | Time-series forecasting with seasonality |
| L07 | Neural networks, the PyTorch training loop |
| L08 | CNNs, transfer learning, computer vision |
| L09 | Sentence embeddings, semantic search, hybrid retrieval |
| L10 | Attention, transformers, LLMs, RAG |

You can ship classical ML systems. You can fine-tune CNNs. You can build semantic search. And as of this notebook, you can build a RAG-powered chat application from scratch.

That's a full-stack production ML engineer's toolkit.

**The lesson Sarah's journey teaches:** every shipping ML system is a chain of decisions about *which problem to solve, which tool to use, and what to measure*. The mathematics is the easy part. The hard part is taste — picking the right tool, being honest about what works, and shipping something that helps real customers.

Welcome to the field. Now go build something.

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## E1 · The hallucination check

A simple safety pattern: after generation, check that every product name mentioned in the answer was actually in the retrieved set. If not, flag the response for review.

In [35]:
import re

def hallucination_check(answer, retrieved):
    """Audit an answer against the retrieved set and the full catalogue."""
    text = answer.lower()
    retrieved_names = set(retrieved['name'])   # the names we actually gave the model for this query
    catalogue_names = set(df['name'])          # every real product name in the catalogue
    # 1) Real catalogue names the answer mentions. Match case-insensitively --
    #    small models often write "rose puff sleeve top" instead of "Rose Puff Sleeve Top".
    mentioned = {n for n in catalogue_names if n.lower() in text}
    # 2) Capitalised multi-word phrases that LOOK like product names but match nothing real.
    #    (Naive heuristic -- in production you'd ask for product IDs, which are easier to verify.)
    phrases = set(re.findall(r'\b[A-Z][a-z]+(?: [A-Z][a-z]+)+\b', answer))
    catalogue_lower = {n.lower() for n in catalogue_names}
    invented = {p for p in phrases
                if p.lower() not in catalogue_lower and not any(p.lower() in n for n in catalogue_lower)}
    return {
        'grounded': sorted(mentioned & retrieved_names),             # real AND retrieved -> fine
        'outside_retrieved': sorted(mentioned - retrieved_names),    # real, but NOT shown to the model -> flag
        'potentially_invented': sorted(invented),                    # not in the catalogue at all -> flag
    }

q = "What's a fancy outfit for a wedding?"
result = rag.ask(q)
print(f"Q: {q}\nA: {result['answer']}\n")
check = hallucination_check(result['answer'], result['retrieved'])  # audit the answer for hallucinations
print(f"Grounded (in retrieved set)     : {check['grounded']}")
print(f"⚠️  Real but NOT retrieved        : {check['outside_retrieved']}")
print(f"⚠️  Potentially invented          : {check['potentially_invented']}")

Q: What's a fancy outfit for a wedding?
A: The best way to look stylish and elegant is to pair your rose puff sleeve top with an ivy slip dress that complements your style perfectly. The ivy slip dress adds a touch of sophistication while keeping things simple and understated.

Grounded (in retrieved set)     : ['Ivy Slip Dress', 'Rose Puff Sleeve Top']
⚠️  Real but NOT retrieved        : []
⚠️  Potentially invented          : []


## E2 · Re-ranking with the LLM

A more sophisticated retrieval step: retrieve a wide top-10 with embeddings, then ask the LLM to re-rank those 10. The LLM brings deeper understanding but is more expensive.

In [36]:
def llm_rerank(query, retrieved, top_k=3):
    """Ask the LLM to pick the most relevant products from a list.

    Subtle bug to avoid: don't put example product IDs in the prompt
    (e.g. 'Format: P0001, P0042') — small models often copy the example IDs
    instead of picking from the actual options. Use a structural example only.
    """
    # Format the wide candidate set as a list (id: name — short description) for the model to rank.
    options = []
    for _, r in retrieved.iterrows():
        options.append(f"  {r['product_id']}: {r['name']} — {r['description'][:80]}")
    options_str = '\n'.join(options)

    # Ask the LLM to act purely as a relevance ranker and output ONLY product IDs.
    messages = [
        {'role': 'system', 'content': 'You are a product-search relevance ranker.'},
        {'role': 'user',   'content': (
            f"Customer query: {query}\n\n"
            f"Product options:\n{options_str}\n\n"
            f"List the {top_k} most relevant product IDs from above, in order of relevance, comma-separated. "
            "Only output the product IDs, nothing else."
        )},
    ]
    response = llm_chat(messages, max_new_tokens=40)
    # Parse product IDs from the response (any P#### pattern in the model's text).
    all_ids = re.findall(r'P\d{4}', response)
    # Filter to IDs that were actually in the candidate set
    # (guards against the model echoing IDs that weren't among the options).
    candidate_ids = set(retrieved['product_id'].tolist())
    valid_ids = [i for i in all_ids if i in candidate_ids]
    return valid_ids[:top_k], response

q = "Something to wear running on a cold morning"
# Re-ranking flow: retrieve WIDE (top-10) with fast embeddings...
retrieved = rag.retrieve(q, top_k=10)
print(f"Initial top-10 by embeddings:")
print(retrieved[['product_id','name']].to_string(index=False))
print()
# ...then let the (slower, smarter) LLM NARROW that list down to the best 3.
ranked_ids, raw = llm_rerank(q, retrieved, top_k=3)
print(f"\nLLM raw response: {raw!r}")
print(f"LLM re-ranked top-3 (filtered to candidates): {ranked_ids}")
# Name the picks
for pid in ranked_ids:
    name = retrieved.loc[retrieved['product_id']==pid, 'name'].iloc[0]
    print(f"  {pid}: {name}")

Initial top-10 by embeddings:
product_id                   name
     P0061   Drift Running Shorts
     P0041 Cloud Running Trainers
     P0014      Frost Linen Shirt
     P0007   Sienna Bodycon Dress
     P0018   Rose Puff Sleeve Top
     P0033    Cinder Biker Jacket
     P0030   Ember Quilted Jacket
     P0031    Verge Bomber Jacket
     P0028  Glacier Puffer Jacket
     P0043       Loam Ankle Boots


LLM raw response: 'P0061, P0041, P0014'
LLM re-ranked top-3 (filtered to candidates): ['P0061', 'P0041', 'P0014']
  P0061: Drift Running Shorts
  P0041: Cloud Running Trainers
  P0014: Frost Linen Shirt


**Look at the result:** the 360M model returned exactly the embeddings' own top three, in the same order — it added no information. That is a real finding, not a bug: a re-ranker is only worth its extra cost if it understands relevance *better* than the retriever. Production systems use a large LLM or a dedicated cross-encoder re-ranker for this step.

The pattern itself — embeddings retrieve a wide candidate set cheaply, a stronger model re-ranks precisely — is what production-grade RAG systems use. Most queries get answered well by retrieval alone; the re-rank is a finishing pass.